# Avatar_ML Colab inference server

Runs the **data plane** of the hybrid system: OpenVoice V2 + MuseTalk 1.5 behind a FastAPI server exposed through a Cloudflare Tunnel. Windows talks to this URL.

## How to use

1. **Runtime → Change runtime type → T4 GPU**.
2. Run all cells, top to bottom. Cell 2 takes 6–10 minutes (it installs Python 3.11 and a lot of ML deps).
3. Copy the printed `COLAB_INFERENCE_URL=https://*.trycloudflare.com` line into your local `.env` on Windows.
4. Keep this tab open — closing it idles the runtime quickly.

When Colab disconnects, restart and re-run all cells, paste the new URL into Windows `.env`, then run `python scripts/create_avatar_profile.py --rehydrate <avatar_id>` on Windows.

## Why this notebook installs Python 3.11 separately

Colab's default kernel is Python **3.12**, but OpenVoice / MeloTTS / parts of MuseTalk's `requirements.txt` fail to build on 3.12 (their `setup.py` was written for 3.10/3.11). We install Python 3.11 with `apt` + `deadsnakes`, create a venv at `/content/py311`, install the model stack there, and run `uvicorn` from that venv. The notebook kernel itself stays on 3.12 — it just orchestrates.

## Two model-inference TODOs

The server code (Cell 3) includes the full wire protocol exactly as Windows expects. Two endpoints — `/avatar/preprocess` and `WS /lipsync/stream` — are marked **TODO** because MuseTalk's internal API varies by version. Each TODO points to the exact MuseTalk file to adapt. The other endpoints — `/healthz`, `/voice/extract_embedding`, `/avatar/upload_cache`, `WS /tts/stream` — are fully implemented.

In [ ]:
# Cell 2 — install Python 3.11, create venv, install model deps, download weights.
import os, sys, subprocess
print('Notebook kernel:', sys.version)
!nvidia-smi | head -n 4

# 1) Install Python 3.11 if missing.
have_311 = subprocess.run(['which', 'python3.11'], capture_output=True).returncode == 0
if not have_311:
    print('Installing Python 3.11...')
    !sudo apt-get update -qq
    !sudo apt-get install -y -qq software-properties-common
    !sudo add-apt-repository -y ppa:deadsnakes/ppa
    !sudo apt-get update -qq
    !sudo apt-get install -y -qq python3.11 python3.11-venv python3.11-dev
!python3.11 --version

# 2) Create the venv if needed.
PY311 = '/content/py311'
if not os.path.exists(f'{PY311}/bin/python'):
    !python3.11 -m venv {PY311}

PY  = f'{PY311}/bin/python'
PIP = f'{PY311}/bin/pip'
!{PIP} install --quiet --upgrade pip

# 3) Server runtime + pinned numpy + torch for CUDA 12.1 (T4).
# opencv-python pinned <4.9 because newer wheels require numpy>=2 (we pin 1.26.4).
!{PIP} install "fastapi>=0.115" "uvicorn[standard]>=0.32" "python-multipart>=0.0.12" "websockets>=13" "Pillow>=10.4" "soundfile>=0.12" "librosa==0.9.1" "huggingface_hub>=0.24" "numpy==1.26.4" "opencv-python==4.8.1.78" 2>&1 | tail -3
!{PIP} install torch==2.1.2 torchaudio==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -3

# 4) Clone the three model repos.
for repo, url in [
    ('OpenVoice', 'https://github.com/myshell-ai/OpenVoice'),
    ('MeloTTS',   'https://github.com/myshell-ai/MeloTTS'),
    ('MuseTalk',  'https://github.com/TMElyralab/MuseTalk'),
]:
    if not os.path.exists(f'/content/{repo}'):
        !git clone --depth 1 {url} /content/{repo}

# 5) OpenVoice + MeloTTS runtime deps. Two smaller groups so a single bad pin
# doesn't roll back the whole list. No --quiet so failures are visible.
print('--- Installing OpenVoice runtime deps ---')
!{PIP} install \
    "Unidecode>=1.3" \
    "openai-whisper" \
    "whisper-timestamped" \
    "faster-whisper" \
    "pydub" \
    "wavmark" \
    "inflect" \
    "eng-to-ipa" \
    "pypinyin" \
    "cn2an" \
    "jieba" \
    "langid" \
    "dominate" 2>&1 | tail -5

print('--- Installing MeloTTS runtime deps ---')
!{PIP} install \
    "mecab-python3>=1.0" \
    "unidic-lite>=1.0" \
    "num2words" \
    "pykakasi" \
    "fugashi" \
    "g2p_en" \
    "anyascii" \
    "jamo" \
    "gruut" \
    "cached_path" \
    "nltk" \
    "transformers==4.27.4" \
    "tokenizers==0.13.3" 2>&1 | tail -5

# 6) Make the cloned repos importable from the venv.
sp = f'{PY311}/lib/python3.11/site-packages'
!echo '/content/OpenVoice' > {sp}/openvoice.pth
!echo '/content/MeloTTS'   > {sp}/melo.pth
!echo '/content/MuseTalk'  > {sp}/musetalk.pth

# 7) MeloTTS needs unidic dictionary at runtime.
!{PY} -m unidic download 2>&1 | tail -3

# 8) Deep sanity-check — exercise the imports the runtime ACTUALLY uses.
print('--- Sanity check: imports the runtime path uses ---')
!{PY} -c "from unidecode import unidecode; print('unidecode: OK')"
!{PY} -c "import cv2; print('cv2: OK', cv2.__version__)"
!{PY} -c "from openvoice.api import ToneColorConverter; print('ToneColorConverter: OK')"
!{PY} -c "from openvoice import se_extractor; print('se_extractor: OK')"
!{PY} -c "import whisper; print('whisper: OK')"
!{PY} -c "from melo.api import TTS; print('MeloTTS TTS: OK')"

# 9) Download OpenVoice V2 weights now — this is what VP2 needs.
OPENVOICE_CKPT = '/content/models/openvoice/checkpoints_v2'
os.makedirs(OPENVOICE_CKPT, exist_ok=True)
!{PY} -c "from huggingface_hub import snapshot_download; snapshot_download(repo_id='myshell-ai/OpenVoiceV2', local_dir='{OPENVOICE_CKPT}')"
print('OpenVoice weights ready. VP2 will work past this point.')

# 10) MuseTalk deps — needed for VP3. Each step wrapped so failures don't
# block the cell. Deliberately NOT installing mmpose (pulls chumpy, fails on
# Python 3.11). MuseTalk lip-sync only needs mmcv.
!{PIP} install --quiet openmim 2>&1 | tail -3 || echo '(openmim install failed; OK to skip for VP2)'
!{PY} -m mim install mmcv==2.1.0 2>&1 | tail -3 || echo '(mmcv install failed; finish manually before VP3)'
!{PIP} install --quiet -r /content/MuseTalk/requirements.txt 2>&1 | tail -3 || echo '(MuseTalk requirements partial; OK to skip for VP2)'

# 11) MuseTalk weights (upstream uses .sh, not .py).
!cd /content/MuseTalk && bash download_weights.sh 2>&1 | tail -5 || echo '(MuseTalk weights skipped; finish manually before VP3)'

# 12) Final MuseTalk import sanity check — proves VP3 path is wired.
print('--- Sanity check: MuseTalk imports the runtime needs ---')
!{PY} -c "from musetalk.utils.utils import load_all_model, get_video_fps, datagen; print('musetalk.utils.utils: OK')" || echo '(musetalk imports failed — VP3 will not work until fixed)'
!{PY} -c "from musetalk.utils.preprocessing import get_landmark_and_bbox, read_imgs, coord_placeholder; print('musetalk.preprocessing: OK')" || echo '(preprocessing imports failed)'
!{PY} -c "from musetalk.utils.blending import get_image_prepare_material, get_image; print('musetalk.blending: OK')" || echo '(blending imports failed)'

print('\n=== Cell 2 done: Python 3.11 venv ready at /content/py311 ===')

In [ ]:
%%writefile /content/server.py
"""Avatar_ML Colab inference server. Runs under /content/py311/bin/python.

Wire protocol mirrors Plan.md §5.5. Do not change message shapes without also
updating the Windows-side ColabWorkerClient.
"""
import os, sys, io, json, base64, time, tarfile, shutil, traceback, pickle, glob
from pathlib import Path

import numpy as np
import soundfile as sf
import librosa
import torch
import cv2
from fastapi import FastAPI, UploadFile, File, Request, Query, WebSocket, WebSocketDisconnect, HTTPException
from fastapi.responses import Response

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OPENVOICE_CKPT = '/content/models/openvoice/checkpoints_v2'
MUSETALK_DIR  = '/content/MuseTalk'
CACHE_ROOT    = '/content/cache'
Path(CACHE_ROOT).mkdir(parents=True, exist_ok=True)
print('[server] Python:', sys.version.split()[0], '| Device:', DEVICE)

if MUSETALK_DIR not in sys.path:
    sys.path.insert(0, MUSETALK_DIR)

# ============================================================================
# Lazy loaders
# ============================================================================
_ov = {'tcc': None, 'melo': None, 'source_se': None}
_mt = {'loaded': False, 'modules': {}}

def lazy_load_openvoice():
    if _ov['tcc'] is not None:
        return _ov
    from openvoice.api import ToneColorConverter
    tcc = ToneColorConverter(f'{OPENVOICE_CKPT}/converter/config.json', device=DEVICE)
    tcc.load_ckpt(f'{OPENVOICE_CKPT}/converter/checkpoint.pth')
    _ov['tcc'] = tcc
    from melo.api import TTS
    _ov['melo'] = TTS(language='EN', device=DEVICE)
    _ov['source_se'] = torch.load(
        f'{OPENVOICE_CKPT}/base_speakers/ses/en-us.pth', map_location=DEVICE
    )
    print('[lazy] OpenVoice V2 + MeloTTS loaded.')
    return _ov

def lazy_load_musetalk():
    if _mt['loaded']:
        return _mt
    from musetalk.utils.utils import load_all_model
    audio_processor, vae, unet, pe = load_all_model()
    vae.vae = vae.vae.half().to(DEVICE)
    unet.model = unet.model.half().to(DEVICE)
    pe = pe.half().to(DEVICE)
    _mt['modules'] = {
        'audio_processor': audio_processor, 'vae': vae, 'unet': unet, 'pe': pe,
    }
    _mt['loaded'] = True
    print('[lazy] MuseTalk loaded.')
    return _mt

# ============================================================================
# Defensive shims around MuseTalk's shifting API
# ============================================================================
def _vae_encode(vae, crop_256):
    """VAE-encode a 256x256 face crop. Method name varies across MuseTalk versions."""
    for name in ('get_latents_for_unet', 'encode_latents', 'encode'):
        fn = getattr(vae, name, None)
        if fn is not None:
            return fn(crop_256)
    raise AttributeError(
        f'VAE has no known encode method. Tried get_latents_for_unet/encode_latents/encode. '
        f'Available: {[m for m in dir(vae) if not m.startswith("_")][:30]}'
    )

def _vae_decode(vae, pred_latents):
    """Decode predicted latents to face crops."""
    for name in ('decode_latents', 'decode'):
        fn = getattr(vae, name, None)
        if fn is not None:
            return fn(pred_latents)
    raise AttributeError(
        f'VAE has no known decode method. Available: {[m for m in dir(vae) if not m.startswith("_")][:30]}'
    )

def _get_image_material(frame, bbox):
    """Normalize get_image_prepare_material return to (mask, crop_box)."""
    from musetalk.utils.blending import get_image_prepare_material
    res = get_image_prepare_material(frame, bbox)
    if isinstance(res, tuple) and len(res) == 2:
        return res
    return res, bbox

def _blend(frame, face, bbox, mask, crop_box):
    """Call get_image with either 5-arg or 4-arg signature."""
    from musetalk.utils.blending import get_image
    try:
        return get_image(frame, face, bbox, mask, crop_box)
    except TypeError:
        return get_image(frame, face, bbox, mask)

def _is_placeholder(bbox):
    """Works for tuple, list, numpy array — coord_placeholder is (0,0,0,0)."""
    try:
        return all(int(x) == 0 for x in bbox)
    except Exception:
        from musetalk.utils.preprocessing import coord_placeholder
        return bbox == coord_placeholder

def _unet_forward(unet, latent_batch, audio_feat):
    """unet.model(...) may return tensor directly or an object with .sample."""
    out = unet.model(latent_batch, 0, encoder_hidden_states=audio_feat)
    return getattr(out, 'sample', out)

def _feature2chunks(audio_processor, feature_array, fps):
    """feature2chunks kwargs vary across MuseTalk versions."""
    try:
        return audio_processor.feature2chunks(feature_array=feature_array, fps=fps)
    except TypeError:
        return audio_processor.feature2chunks(feature_array, fps)

def _unet_dtype(unet):
    """Best-effort: figure out the dtype of the UNet model for casting inputs."""
    try:
        return unet.model.dtype
    except AttributeError:
        try:
            return next(unet.model.parameters()).dtype
        except Exception:
            return torch.float16

# ============================================================================
# FastAPI app
# ============================================================================
app = FastAPI(title='Avatar_ML Colab inference server')

@app.get('/healthz')
def healthz():
    free_vram_mb = None
    if torch.cuda.is_available():
        free, _ = torch.cuda.mem_get_info()
        free_vram_mb = int(free / 1e6)
    return {
        'status': 'ok',
        'python': sys.version.split()[0],
        'gpu': torch.cuda.is_available(),
        'openvoice_loaded': _ov['tcc'] is not None,
        'musetalk_loaded': _mt['loaded'],
        'free_vram_mb': free_vram_mb,
    }

# ============================================================================
# Voice — fully implemented
# ============================================================================
@app.post('/voice/extract_embedding')
async def extract_voice_embedding(audio: UploadFile = File(...)):
    try:
        ov = lazy_load_openvoice()
        upload_dir = Path('/content/uploads/voice')
        upload_dir.mkdir(parents=True, exist_ok=True)
        wav_path = upload_dir / (audio.filename or 'voice.wav')
        wav_path.write_bytes(await audio.read())
        from openvoice import se_extractor
        target_se, _name = se_extractor.get_se(str(wav_path), ov['tcc'], vad=False)
        buf = io.BytesIO()
        np.save(buf, target_se.detach().cpu().numpy())
        return Response(content=buf.getvalue(), media_type='application/octet-stream')
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=f'{type(e).__name__}: {e}')

# ============================================================================
# Avatar preprocessing
# ============================================================================
def musetalk_preprocess(video_path: Path, out_dir: Path, bbox_shift: int = 0):
    """Run MuseTalk preprocessing on video_path. Writes artifacts into out_dir/.

    Output layout:
      out_dir/coords.pkl
      out_dir/mask_coords.pkl
      out_dir/latents.pt
      out_dir/full_imgs/NNNNNNNN.png
      out_dir/mask/NNNNNNNN.png
      out_dir/avatar_info.json
    """
    from musetalk.utils.preprocessing import get_landmark_and_bbox
    from musetalk.utils.utils import get_video_fps

    mt = lazy_load_musetalk()
    vae = mt['modules']['vae']

    out_dir = Path(out_dir)
    full_imgs_dir = out_dir / 'full_imgs'
    mask_dir = out_dir / 'mask'
    out_dir.mkdir(parents=True, exist_ok=True)
    full_imgs_dir.mkdir(exist_ok=True)
    mask_dir.mkdir(exist_ok=True)

    fps = get_video_fps(str(video_path))
    print(f'[preprocess] video fps={fps}')

    cap = cv2.VideoCapture(str(video_path))
    img_paths = []
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        p = str(full_imgs_dir / f'{idx:08d}.png')
        cv2.imwrite(p, frame)
        img_paths.append(p)
        idx += 1
    cap.release()
    print(f'[preprocess] extracted {len(img_paths)} frames -> {full_imgs_dir}')

    print(f'[preprocess] running face detection (slow part)...')
    coord_list, frame_list = get_landmark_and_bbox(img_paths, upperbondrange=bbox_shift)
    n_detected = sum(1 for b in coord_list if not _is_placeholder(b))
    print(f'[preprocess] detected faces in {n_detected}/{len(coord_list)} frames')

    print(f'[preprocess] VAE-encoding {n_detected} face crops...')
    input_latent_list = []
    for bbox, frame in zip(coord_list, frame_list):
        if _is_placeholder(bbox):
            continue
        x1, y1, x2, y2 = bbox
        crop_frame = frame[y1:y2, x1:x2]
        crop_frame = cv2.resize(crop_frame, (256, 256), interpolation=cv2.INTER_LANCZOS4)
        latent = _vae_encode(vae, crop_frame)
        input_latent_list.append(latent)
    print(f'[preprocess] encoded {len(input_latent_list)} face latents')

    print(f'[preprocess] generating masks...')
    mask_coords_list = []
    for i, (bbox, frame) in enumerate(zip(coord_list, frame_list)):
        if _is_placeholder(bbox):
            mask_coords_list.append((0, 0, 0, 0))
            continue
        mask, crop_box = _get_image_material(frame, bbox)
        cv2.imwrite(str(mask_dir / f'{i:08d}.png'), mask)
        mask_coords_list.append(crop_box)
    print(f'[preprocess] wrote {len(os.listdir(mask_dir))} masks')

    with open(out_dir / 'coords.pkl', 'wb') as f:
        pickle.dump(coord_list, f)
    with open(out_dir / 'mask_coords.pkl', 'wb') as f:
        pickle.dump(mask_coords_list, f)
    torch.save(input_latent_list, out_dir / 'latents.pt')
    with open(out_dir / 'avatar_info.json', 'w') as f:
        json.dump({'bbox_shift': bbox_shift, 'fps': fps,
                   'num_frames': len(img_paths)}, f)
    print(f'[preprocess] done')

@app.post('/avatar/preprocess')
async def preprocess_avatar(video: UploadFile = File(...)):
    try:
        upload_dir = Path('/content/uploads/avatar')
        upload_dir.mkdir(parents=True, exist_ok=True)
        mp4_path = upload_dir / (video.filename or 'face.mp4')
        mp4_path.write_bytes(await video.read())

        ts = int(time.time())
        tmp_root = Path(CACHE_ROOT) / f'_preproc_{ts}'
        cache_dir = tmp_root / 'cache'
        musetalk_preprocess(mp4_path, cache_dir)

        tar_path = Path(f'/tmp/avatar_cache_{ts}.tar.gz')
        with tarfile.open(tar_path, 'w:gz') as tar:
            tar.add(cache_dir, arcname='cache')
        body = tar_path.read_bytes()
        tar_path.unlink(missing_ok=True)
        shutil.rmtree(tmp_root, ignore_errors=True)

        return Response(content=body, media_type='application/gzip')
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=f'{type(e).__name__}: {e}')

@app.post('/avatar/upload_cache')
async def upload_avatar_cache(request: Request, avatar_id: str = Query(...)):
    body = await request.body()
    dest = Path(CACHE_ROOT) / avatar_id
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True, exist_ok=True)
    tar_path = dest / 'cache.tar.gz'
    tar_path.write_bytes(body)
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(dest)
    return {'avatar_id': avatar_id, 'extracted_to': str(dest), 'size_bytes': len(body)}

# ============================================================================
# TTS — fully implemented
# ============================================================================
def _to_24khz_int16(wav_path: str) -> bytes:
    data, sr = sf.read(wav_path, dtype='int16')
    if data.ndim > 1:
        data = data[:, 0]
    if sr != 24000:
        f = data.astype(np.float32) / 32768.0
        f = librosa.resample(f, orig_sr=sr, target_sr=24000)
        data = np.clip(f * 32768.0, -32768, 32767).astype(np.int16)
    return data.tobytes()

def _synthesize_pcm(text: str, target_se_b64: str) -> bytes:
    ov = lazy_load_openvoice()
    arr = np.load(io.BytesIO(base64.b64decode(target_se_b64)))
    target_se = torch.from_numpy(arr).to(DEVICE)
    if target_se.dim() == 1:
        target_se = target_se.unsqueeze(0).unsqueeze(-1)
    src_path = '/tmp/openvoice_src.wav'
    out_path = '/tmp/openvoice_out.wav'
    spk_ids = ov['melo'].hps.data.spk2id
    spk_key = ('EN-US' if 'EN-US' in spk_ids
               else 'EN-Default' if 'EN-Default' in spk_ids
               else next(iter(spk_ids)))
    ov['melo'].tts_to_file(text, spk_ids[spk_key], src_path, speed=1.0)
    ov['tcc'].convert(
        audio_src_path=src_path,
        src_se=ov['source_se'],
        tgt_se=target_se,
        output_path=out_path,
    )
    return _to_24khz_int16(out_path)

@app.websocket('/tts/stream')
async def tts_stream(ws: WebSocket):
    await ws.accept()
    try:
        payload = json.loads(await ws.receive_text())
        pcm_bytes = _synthesize_pcm(payload['text'], payload['embedding_b64'])
        chunk_sz = 9600
        pts = 0.0
        for i in range(0, len(pcm_bytes), chunk_sz):
            chunk = pcm_bytes[i:i + chunk_sz]
            await ws.send_text(json.dumps({
                'type': 'pcm', 'pts': pts,
                'data': base64.b64encode(chunk).decode(),
            }))
            pts += (len(chunk) // 2) / 24000.0
        await ws.send_text(json.dumps({'type': 'end'}))
    except WebSocketDisconnect:
        pass
    except Exception as e:
        traceback.print_exc()
        try:
            await ws.send_text(json.dumps({'type': 'error', 'message': f'{type(e).__name__}: {e}'}))
        except Exception:
            pass

# ============================================================================
# Lip-sync
# ============================================================================
def _ping_pong_index(i: int, n: int) -> int:
    if n <= 1:
        return 0
    period = 2 * (n - 1)
    i = i % period
    return i if i < n else period - i

def _run_musetalk(avatar_id: str, pcm_bytes: bytes, fps: int = 25):
    """Generator yielding (frame_idx, jpeg_bytes) for the full audio."""
    from musetalk.utils.utils import datagen
    from musetalk.utils.preprocessing import read_imgs

    mt = lazy_load_musetalk()
    audio_processor = mt['modules']['audio_processor']
    vae    = mt['modules']['vae']
    unet   = mt['modules']['unet']
    pe     = mt['modules']['pe']

    cache_dir = Path(CACHE_ROOT) / avatar_id / 'cache'
    if not cache_dir.exists():
        raise FileNotFoundError(
            f'No avatar cache at {cache_dir}. Re-upload via /avatar/upload_cache.'
        )

    with open(cache_dir / 'coords.pkl', 'rb') as f:
        coord_list = pickle.load(f)
    with open(cache_dir / 'mask_coords.pkl', 'rb') as f:
        mask_coords_list = pickle.load(f)
    input_latent_list = torch.load(cache_dir / 'latents.pt')

    full_imgs_paths = sorted(glob.glob(str(cache_dir / 'full_imgs' / '*.png')))
    mask_paths      = sorted(glob.glob(str(cache_dir / 'mask' / '*.png')))
    frame_list = read_imgs(full_imgs_paths)
    mask_list  = read_imgs(mask_paths)
    print(f'[lipsync] loaded cache: {len(frame_list)} frames, '
          f'{len(input_latent_list)} latents')

    # OpenVoice outputs 24 kHz; Whisper (inside audio_processor) expects 16 kHz.
    # Explicitly resample here so we don't depend on audio_processor handling it.
    audio_path = '/tmp/lipsync_input.wav'
    arr = np.frombuffer(pcm_bytes, dtype=np.int16).astype(np.float32) / 32768.0
    arr_16k = librosa.resample(arr, orig_sr=24000, target_sr=16000)
    sf.write(audio_path, (arr_16k * 32768.0).clip(-32768, 32767).astype(np.int16),
             16000, subtype='PCM_16')

    print('[lipsync] extracting whisper features...')
    whisper_feature = audio_processor.audio2feat(audio_path)
    whisper_chunks  = _feature2chunks(audio_processor, whisper_feature, fps)
    print(f'[lipsync] {len(whisper_chunks)} audio chunks')

    n_latents = len(input_latent_list)
    batch_size = 8
    gen = datagen(
        whisper_chunks=whisper_chunks,
        vae_encode_latents=input_latent_list,
        batch_size=batch_size,
        delay_frame=0,
    )

    unet_dtype = _unet_dtype(unet)

    frame_idx = 0
    for whisper_batch, latent_batch in gen:
        audio_feature_batch = torch.from_numpy(whisper_batch).to(
            device=DEVICE, dtype=unet_dtype
        )
        audio_feature_batch = pe(audio_feature_batch)
        latent_batch = latent_batch.to(dtype=unet_dtype)

        pred_latents = _unet_forward(unet, latent_batch, audio_feature_batch)
        recon = _vae_decode(vae, pred_latents)

        for res_face in recon:
            cur_idx = _ping_pong_index(frame_idx, n_latents)
            ori_frame = frame_list[cur_idx].copy()
            bbox = coord_list[cur_idx]

            if _is_placeholder(bbox):
                combine = ori_frame
            else:
                x1, y1, x2, y2 = bbox
                res_face_resized = cv2.resize(
                    res_face.astype(np.uint8), (x2 - x1, y2 - y1)
                )
                combine = _blend(
                    ori_frame, res_face_resized, bbox,
                    mask_list[cur_idx], mask_coords_list[cur_idx],
                )

            ok, jpeg_buf = cv2.imencode(
                '.jpg', combine, [cv2.IMWRITE_JPEG_QUALITY, 85]
            )
            if not ok:
                raise RuntimeError(f'cv2.imencode failed at frame {frame_idx}')
            yield frame_idx, jpeg_buf.tobytes()
            frame_idx += 1

@app.websocket('/lipsync/stream')
async def lipsync_stream(ws: WebSocket):
    await ws.accept()
    try:
        first = json.loads(await ws.receive_text())
        avatar_id = first['avatar_id']

        pcm_buf = []
        while True:
            m = json.loads(await ws.receive_text())
            if m.get('type') == 'end':
                break
            if m.get('type') == 'pcm':
                pcm_buf.append(base64.b64decode(m['data']))
        full_pcm = b''.join(pcm_buf)
        print(f'[lipsync] received {len(full_pcm)} bytes PCM '
              f'(~{len(full_pcm)/2/24000:.1f}s audio)')

        for frame_idx, jpeg_bytes in _run_musetalk(avatar_id, full_pcm, fps=25):
            await ws.send_text(json.dumps({
                'type': 'frame',
                'frame_idx': frame_idx,
                'pts': frame_idx / 25.0,
                'jpeg_b64': base64.b64encode(jpeg_bytes).decode(),
            }))
        await ws.send_text(json.dumps({'type': 'end'}))
    except WebSocketDisconnect:
        pass
    except Exception as e:
        traceback.print_exc()
        try:
            await ws.send_text(json.dumps({'type': 'error', 'message': f'{type(e).__name__}: {e}'}))
        except Exception:
            pass

if __name__ == '__main__':
    import uvicorn
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')


In [ ]:
# Cell P (optional probe) — inspect MuseTalk's actual API.
#
# Run this ONLY if VP3 (avatar profile) fails, or if you want pre-flight
# verification. It loads MuseTalk into VRAM (slow ~30 s on first call) and
# prints the function signatures + method names the server.py defensive shims
# rely on. Paste the output into the chat if any expected symbol is missing.
import subprocess
result = subprocess.run(
    ['/content/py311/bin/python', '-'],
    input=r'''
import inspect, json, traceback
try:
    from musetalk.utils.utils import load_all_model, datagen, get_video_fps
    from musetalk.utils.preprocessing import get_landmark_and_bbox, coord_placeholder, read_imgs
    from musetalk.utils.blending import get_image_prepare_material, get_image
    out = {
        "coord_placeholder": repr(coord_placeholder),
        "datagen": str(inspect.signature(datagen)),
        "get_landmark_and_bbox": str(inspect.signature(get_landmark_and_bbox)),
        "read_imgs": str(inspect.signature(read_imgs)),
        "get_image_prepare_material": str(inspect.signature(get_image_prepare_material)),
        "get_image": str(inspect.signature(get_image)),
    }
    ap, vae, unet, pe = load_all_model()
    out["vae_methods"]  = [m for m in dir(vae)  if not m.startswith("_")][:30]
    out["unet_methods"] = [m for m in dir(unet) if not m.startswith("_")][:30]
    out["ap_methods"]   = [m for m in dir(ap)   if not m.startswith("_")][:30]
    out["vae_has_get_latents_for_unet"] = hasattr(vae, "get_latents_for_unet")
    out["vae_has_decode_latents"]       = hasattr(vae, "decode_latents")
    out["ap_has_audio2feat"]            = hasattr(ap,  "audio2feat")
    out["ap_has_feature2chunks"]        = hasattr(ap,  "feature2chunks")
    print(json.dumps(out, indent=2))
except Exception:
    traceback.print_exc()
''',
    capture_output=True, text=True, timeout=180,
)
print(result.stdout)
if result.returncode != 0 or result.stderr:
    print('--- STDERR ---'); print(result.stderr)

In [ ]:
# Cell 4 — start (or restart) the Cloudflare tunnel + uvicorn under py311.
#
# Re-running this cell is the canonical way to restart the server. It kills any
# previous uvicorn / cloudflared on port 8000 first, then verifies uvicorn is
# actually serving /healthz before printing READY — so a uvicorn crash never
# masquerades as a tunnel URL.
import subprocess, re, threading, time, os
from collections import deque

if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

PORT = 8000

!fuser -k {PORT}/tcp 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
time.sleep(2)

tunnel_url_ref = {'url': None}
uvicorn_tail   = deque(maxlen=40)   # last 40 lines for diagnosing crashes

def run_tunnel():
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print('[tunnel]', line, end='')
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m and not tunnel_url_ref['url']:
            tunnel_url_ref['url'] = m.group(0)

def run_uvicorn():
    proc = subprocess.Popen(
        ['/content/py311/bin/python', '-m', 'uvicorn',
         'server:app', '--host', '0.0.0.0', '--port', str(PORT)],
        cwd='/content',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        uvicorn_tail.append(line.rstrip())
        print('[uvicorn]', line, end='')

threading.Thread(target=run_tunnel,  daemon=True).start()
threading.Thread(target=run_uvicorn, daemon=True).start()
print(f'Booting uvicorn (port {PORT}) and cloudflared quick-tunnel ...')

# Wait for tunnel URL.
deadline = time.time() + 180
while time.time() < deadline and not tunnel_url_ref['url']:
    time.sleep(1)

if not tunnel_url_ref['url']:
    print('Tunnel URL not detected within 180s. Re-run this cell.')
else:
    # Tunnel up — but is uvicorn actually serving? Hit /healthz before declaring ready.
    import urllib.request, json as _json
    healthz_ok = False
    last_err = None
    for _ in range(30):  # ~60s
        try:
            req = urllib.request.Request(tunnel_url_ref['url'] + '/healthz')
            with urllib.request.urlopen(req, timeout=3) as r:
                body = r.read().decode()
                _json.loads(body)  # validate JSON
                healthz_ok = True
                break
        except Exception as e:
            last_err = e
        time.sleep(2)

    if healthz_ok:
        print()
        print('=' * 60)
        print(f'COLAB_INFERENCE_URL={tunnel_url_ref["url"]}')
        print('=' * 60)
        print('READY. Copy the line above into your local .env on Windows.')
    else:
        print()
        print('!' * 60)
        print(f'Tunnel is up at {tunnel_url_ref["url"]} BUT /healthz is not responding.')
        print(f'Last error: {last_err}')
        print('Likely cause: uvicorn crashed at boot. Recent uvicorn output:')
        print('!' * 60)
        for line in uvicorn_tail:
            print(' ', line)

## Finishing the two TODOs

Both stubs live in **`/content/server.py`** (written by Cell 3).

1. After Cell 2 finishes, open `/content/MuseTalk/scripts/realtime_inference.py` in Colab's file browser.
2. Locate the `Avatar` class. Its `__init__` does preprocessing; its `inference` method does the per-audio-chunk lip-sync.
3. Port the relevant logic into `preprocess_avatar` and `lipsync_stream` in `/content/server.py` (use `%%writefile -a /content/server.py` from a new cell to append, or edit the file directly via Colab's editor).
4. **Re-run Cell 4** — it kills the old uvicorn, starts a fresh one, and prints the new tunnel URL.

## Restart cheat-sheet

- **Restart uvicorn after editing `/content/server.py`:** re-run **Cell 4**.
- **Lost the kernel:** run all cells from the top.
- **Tunnel URL changed:** copy the new `COLAB_INFERENCE_URL=...` line from Cell 4 into Windows `.env`.